# 04 Eval + Validation (LoRA)

Validate LoRA quality after training and track regressions.

This notebook focuses on local-first validation:

1. Resolve repo root and load config
2. Preflight checks (adapter path, prompts, scripts)
3. Quick sample generation sanity check
4. Run rubric evaluation (`scripts/eval/eval_rubric.py`)
5. Inspect `outputs/eval/` summaries and trends

For heavy full training/DPO compute, prefer AutoDL GPU runtime.

In [ ]:
from pathlib import Path
import os
import json
from typing import Any


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs" / "qlora_config.yaml").is_file() and (cand / "scripts" / "eval" / "eval_rubric.py").is_file():
            return cand
    raise FileNotFoundError("Could not find repo root with configs/qlora_config.yaml and scripts/eval/eval_rubric.py")


REPO = find_repo_root()
os.chdir(REPO)
print("REPO:", REPO)

In [ ]:
import yaml

cfg_path = Path("configs/qlora_config.yaml")
with cfg_path.open("r", encoding="utf-8") as fh:
    cfg: dict[str, Any] = yaml.safe_load(fh) or {}

train_cfg = cfg.get("training", {})
data_cfg = cfg.get("data", {})
eval_cfg = cfg.get("eval", {})

output_dir = Path(train_cfg.get("output_dir", "outputs/jinyong-qlora"))
adapter_dir = output_dir / "adapter"
merged_dir = output_dir / "merged"
prompts_path = Path(eval_cfg.get("prompts_jsonl", "scripts/eval/prompts_v2_typed20.jsonl"))

print("output_dir:", output_dir)
print("adapter_dir exists:", adapter_dir.is_dir())
print("merged_dir exists:", merged_dir.is_dir())
print("eval prompts exists:", prompts_path.is_file())
print("instruction_jsonl:", data_cfg.get("instruction_jsonl", "(missing)"))
print("judge_model:", eval_cfg.get("judge_model", "gpt-4o"))

## Quick LoRA sanity generation (no rubric)

This runs one local generation through `scripts/infer/inference.py` to check whether the adapter can produce output before expensive evaluation.

- If your Mac is slow on 7B generation, keep `--max-new-tokens` small.
- If adapter is missing, run training first (or copy adapters from AutoDL).

In [ ]:
import subprocess
import sys

quick_cmd = [
    sys.executable,
    "scripts/infer/inference.py",
    "--config",
    "configs/qlora_config.yaml",
    "--prompt",
    "以金庸风格写一段华山夜战，约120字。",
    "--max-new-tokens",
    "160",
    "--temperature",
    "0.7",
]
print("Running:", " ".join(quick_cmd))
subprocess.run(quick_cmd, check=False)

## Rubric evaluation run

This executes the automated 5-dim rubric evaluation and writes:

- `outputs/eval/<run_id>/generations.jsonl`
- `outputs/eval/<run_id>/summary.json`
- `outputs/eval/eval_results.jsonl`

Set `OPENAI_API_KEY` in your environment before running the cell.

In [ ]:
import subprocess
import sys
from datetime import datetime

run_id = f"local_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

cmd = [
    sys.executable,
    "scripts/eval/eval_rubric.py",
    "--config",
    "configs/qlora_config.yaml",
    "--run-id",
    run_id,
]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, check=False)
print("exit code:", res.returncode)
print("run_id:", run_id)

In [ ]:
from pathlib import Path
import json

summary_hist = Path("outputs/eval/summary_history.jsonl")
if not summary_hist.is_file():
    print("No summary history yet. Run the rubric eval cell first.")
else:
    rows = [json.loads(line) for line in summary_hist.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"Total eval runs: {len(rows)}")
    for row in rows[-3:]:
        print("-", row.get("run_id"), "overall_avg=", row.get("overall_avg"), "gate_passed=", row.get("gate_passed"))

## Optional: local DPO quick cycle (small scale)

Use this only after base SFT validation passes.

```bash
python scripts/dpo/build_preference_pairs.py --config configs/qlora_config.yaml --max-prompts 1
python scripts/train/train_dpo.py --config configs/qlora_config.yaml
python scripts/eval/eval_rubric.py --config configs/qlora_config.yaml --run-id dpo_local_test
```

For full DPO runs, prefer AutoDL GPU.